In [1]:
import numpy as np
import pandas as pd

# Cấu hình hiển thị
pd.set_option("display.max_columns", None)
pd.options.display.float_format = "{:,.1f}".format

# 1. Tải tập dữ liệu
df = pd.read_csv("Crimes_-_2001_to_Present.csv")
n_raw = len(df)

# 2. Chuyển đổi các cột ngày tháng sang kiểu DateTime
DATE_FORMAT = "%m/%d/%Y %I:%M:%S %p"

for col in ["Date", "Updated On"]:
    df[col] = pd.to_datetime(
        df[col],
        format=DATE_FORMAT,
        errors="coerce"
    )

# Phương án dự phòng nếu dữ liệu chứa định dạng ngày khác
for col in ["Date", "Updated On"]:
    if df[col].isna().mean() > 0.01:
        original = pd.read_csv(
            "Crimes_-_2001_to_Present.csv",
            usecols=[col]
        )[col]

        df[col] = pd.to_datetime(
            original,
            errors="coerce"
        )

n_date_failed = df["Date"].isna().sum()

# 3. Lọc dữ liệu từ năm 2020 trở đi
df = df[df["Date"].dt.year >= 2020].copy()

# 4. Chuẩn hóa cột loại tội phạm (Primary Type)
df["Primary Type"] = (
    df["Primary Type"]
    .astype("string")
    .str.strip()
)

df["Primary Type"] = df["Primary Type"].replace({
    "NON - CRIMINAL": "NON-CRIMINAL"
})

# 5. Thay thế các giá trị hành chính không hợp lệ
# Đổi giá trị 0 thành NaN
for col in ["Community Area", "Ward", "District"]:
    if col in df.columns:
        df[col] = df[col].replace(0, np.nan)

# 6. Chuyển đổi và xử lý các cột tọa độ
coordinate_cols = [
    "Latitude",
    "Longitude",
    "X Coordinate",
    "Y Coordinate"
]

for col in coordinate_cols:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

# Đổi tọa độ X, Y bằng 0 thành NaN
df[["X Coordinate", "Y Coordinate"]] = (
    df[["X Coordinate", "Y Coordinate"]]
    .replace(0, np.nan)
)

# Ranh giới địa lý hợp lệ của thành phố Chicago
CHI_LAT_MIN, CHI_LAT_MAX = 41.55, 42.15
CHI_LON_MIN, CHI_LON_MAX = -88.05, -87.40

# Xác định các bản ghi có tọa độ không hợp lệ
bad_coord = (
    df["Latitude"].isna()
    | df["Longitude"].isna()
    | (df["Latitude"] < CHI_LAT_MIN)
    | (df["Latitude"] > CHI_LAT_MAX)
    | (df["Longitude"] < CHI_LON_MIN)
    | (df["Longitude"] > CHI_LON_MAX)
)

n_bad_coord = int(bad_coord.sum())

# Thay thế các tọa độ không hợp lệ bằng NaN
df.loc[
    bad_coord,
    ["Latitude", "Longitude", "X Coordinate", "Y Coordinate"]
] = np.nan

# 7. Xử lý các giá trị bị khuyết (Missing values)
n_case_number_missing = int(
    df["Case Number"].isna().sum()
)

df["Location Description"] = (
    df["Location Description"]
    .fillna("UNKNOWN")
)

df["Case Number"] = (
    df["Case Number"]
    .fillna("UNKNOWN")
)

# 8. Loại bỏ các bản ghi bị trùng lặp theo mã ID
n_duplicates = int(
    df["ID"].duplicated().sum()
)

df = df.drop_duplicates(
    subset=["ID"]
)

# 9. In báo cáo tóm tắt kết quả làm sạch dữ liệu
print("--- KẾT QUẢ DỮ LIỆU SẠCH ---")

print("Số dòng gốc:", f"{n_raw:,}")
print("Số dòng Date parse lỗi:", f"{n_date_failed:,}")
print("Số dòng còn lại từ năm 2020:", f"{len(df):,}")
print("Số cột:", len(df.columns))

print(
    "Số bản ghi ID trùng lặp đã xóa:",
    f"{n_duplicates:,}"
)

print(
    "Số Case Number bị trống:",
    f"{n_case_number_missing:,}"
)

print(
    "Số bản ghi có tọa độ không hợp lệ:",
    f"{n_bad_coord:,}"
)

print(
    "Các giá trị Primary Type:",
    sorted(df["Primary Type"].dropna().unique().tolist())
)

# 10. Lưu tập dữ liệu đã làm sạch ra file CSV
df.to_csv(
    "Chicago_Crimes_2020_Present_Cleaned.csv",
    index=False
)

--- KẾT QUẢ DỮ LIỆU SẠCH ---
Số dòng gốc: 7,842,937
Số dòng Date parse lỗi: 0
Số dòng còn lại từ năm 2020: 790,069
Số cột: 22
Số bản ghi ID trùng lặp đã xóa: 0
Số Case Number bị trống: 0
Số bản ghi có tọa độ không hợp lệ: 15,782
Các giá trị Primary Type: ['ARSON', 'ASSAULT', 'BATTERY', 'BURGLARY', 'CONCEALED CARRY LICENSE VIOLATION', 'CRIM SEXUAL ASSAULT', 'CRIMINAL DAMAGE', 'CRIMINAL SEXUAL ASSAULT', 'CRIMINAL TRESPASS', 'DECEPTIVE PRACTICE', 'GAMBLING', 'HOMICIDE', 'HUMAN TRAFFICKING', 'INTERFERENCE WITH PUBLIC OFFICER', 'INTIMIDATION', 'KIDNAPPING', 'LIQUOR LAW VIOLATION', 'MOTOR VEHICLE THEFT', 'NARCOTICS', 'NON-CRIMINAL', 'OBSCENITY', 'OFFENSE INVOLVING CHILDREN', 'OTHER NARCOTIC VIOLATION', 'OTHER OFFENSE', 'PROSTITUTION', 'PUBLIC INDECENCY', 'PUBLIC PEACE VIOLATION', 'RITUALISM', 'ROBBERY', 'SEX OFFENSE', 'STALKING', 'THEFT', 'WEAPONS VIOLATION']
